# 🏢 WorkFlow Arena — GRPO Training with TRL + Unsloth

**Meta PyTorch OpenEnv Hackathon — Round 2 Finale**  
**Theme 3.1 + Scaler AI Labs Sub-theme: Multi-App Enterprise Workflow**

This notebook trains a small LLM (Qwen3-1.7B) to orchestrate multi-app enterprise workflows via the WorkFlow Arena environment using GRPO (Group Relative Policy Optimization).

**Hardware**: GPU recommended (Colab free T4 works). ~40 min for a short training run.

**What this notebook does**:
1. Installs TRL (with OpenEnv support), Unsloth, trackio, vLLM
2. Connects to the deployed WorkFlow Arena HF Space
3. Runs rollouts with a Qwen model as the agent
4. Trains via GRPO with multi-component rewards
5. Plots reward improvement curves (the 20% score criterion)
6. Saves the fine-tuned model to HuggingFace Hub

## 1. Install Dependencies

In [ ]:
!pip install -Uq git+https://github.com/huggingface/trl.git git+https://github.com/meta-pytorch/OpenEnv.git trackio vllm==0.10.2 bitsandbytes matplotlib

## 2. HuggingFace Login (needed to save fine-tuned model)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Install WorkFlow Arena Client

Since our environment lives on HF Spaces, we can install it as a pip package directly from the Space repo.

In [ ]:
# Connect to the WorkFlow Arena HF Space
WORKFLOW_ARENA_URL = "https://jaydeepshah2025-workflow-arena.hf.space"

import httpx

# Verify the space is reachable
health = httpx.get(f"{WORKFLOW_ARENA_URL}/health", timeout=10)
print(f"Health: {health.json()}")

tasks = httpx.get(f"{WORKFLOW_ARENA_URL}/tasks", timeout=10).json()
print(f"Available workflows: {len(tasks['tasks'])}")
for t in tasks['tasks']:
    print(f"  {t['difficulty']}: {t['name']} ({t['num_required_actions']} actions)")

## 4. Environment Client

A lightweight client that talks to our deployed WorkFlow Arena.

In [ ]:
import httpx
import json as json_lib

class WorkFlowClient:
    def __init__(self, base_url=WORKFLOW_ARENA_URL):
        self.base_url = base_url
        self.session_id = None
        self.http = httpx.Client(timeout=60)

    def reset(self, task_name="employee_onboarding"):
        r = self.http.post(f"{self.base_url}/reset", json={"task_name": task_name}).json()
        self.session_id = r["session_id"]
        return r

    def step(self, message: str):
        r = self.http.post(
            f"{self.base_url}/step",
            json={"session_id": self.session_id, "message": message},
        ).json()
        return r

    def close(self):
        self.http.close()

# Quick test
client = WorkFlowClient()
r = client.reset("employee_onboarding")
print(r['observation']['content'][:500])
client.close()

## 5. Load the Model (Qwen3-1.7B)

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

## 6. System Prompt

In [ ]:
SYSTEM_PROMPT = """You are an enterprise AI agent that orchestrates multi-app business workflows.
You must execute API calls across Gmail, Slack, Jira, HRIS, CRM, Deploy, and Finance systems.

Respond with ONLY a JSON object:
{
  "calls": [
    {
      "app": "gmail|slack|jira|hris|crm|deploy|finance",
      "method": "<method_name>",
      "params": {"key": "value"},
      "reasoning": "<explain WHY this call is needed>"
    }
  ]
}

Available APIs:
- gmail.create_account(email), gmail.send_email(to, subject, body)
- slack.add_user(user_id, channels), slack.send_message(channel, text)
- jira.create_ticket(title, ticket_type, priority, assignee), jira.update_ticket(ticket_id, status)
- hris.create_employee(emp_id, name, email, dept, start_date), hris.assign_equipment(emp_id, equipment)
- crm.update_customer(customer_id, status, tier), crm.create_support_ticket(customer_id, issue, assignee)
- deploy.service(service, version), deploy.rollback(service), deploy.update_status_page(status)
- finance.submit_expense(emp_id, amount, category, description), finance.approve_expense(expense_id, approver)

Complete the workflow by making the correct API calls in the right priority order."""

## 7. Rollout Function (Agent interacts with environment)

In [ ]:
from collections import defaultdict
from trl.experimental.openenv import generate_rollout_completions

MAX_STEPS = 8  # Limit steps per episode for training speed

def rollout_once(trainer, client, dataset_prompt, max_turns=MAX_STEPS):
    """Run one complete workflow episode."""
    # Pick a random workflow
    import random
    task_name = random.choice([
        "employee_onboarding",
        "expense_approval",
        "customer_support",
    ])

    result = client.reset(task_name)
    observation = result["observation"]["content"]

    prompt_ids = []
    completion_ids = []
    logprobs = []
    rewards = []
    completed_per_step = []
    api_success_per_step = []

    for turn in range(max_turns):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": observation[:3000]},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False, enable_thinking=False,
        )

        rollout_outputs = generate_rollout_completions(trainer, [prompt_text])[0]
        prompt_ids.extend(rollout_outputs["prompt_ids"])
        completion_ids.extend(rollout_outputs["completion_ids"])
        logprobs.extend(rollout_outputs["logprobs"])
        completion_text = rollout_outputs.get("text") or tokenizer.decode(
            rollout_outputs["completion_ids"], skip_special_tokens=True
        )

        step_data = client.step(completion_text)
        reward = float(step_data.get("reward", 0) or 0)
        rewards.append(reward)

        info = step_data.get("info", {})
        completed_per_step.append(info.get("completed_actions", 0))
        api_success_per_step.append(info.get("api_calls_successful", 0))

        if step_data.get("done", False):
            break
        observation = step_data["observation"]["content"]

    total_reward = sum(rewards)
    final_completed = completed_per_step[-1] if completed_per_step else 0
    final_api_success = api_success_per_step[-1] if api_success_per_step else 0

    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "total_reward": total_reward,
        "completed_actions": final_completed,
        "api_success": final_api_success,
        "task_name": task_name,
    }

## 8. Rollout Function for GRPOTrainer

In [ ]:
client = WorkFlowClient()

def rollout_func(prompts, trainer=None):
    """Called by GRPOTrainer each training step."""
    episode_prompt_ids = []
    episode_completion_ids = []
    episode_logprobs = []
    total_rewards = []
    completed_rewards = []
    api_rewards = []

    for prompt_text in prompts:
        episode = rollout_once(trainer, client, prompt_text)
        episode_prompt_ids.append(episode["prompt_ids"])
        episode_completion_ids.append(episode["completion_ids"])
        episode_logprobs.append(episode["logprobs"])
        total_rewards.append(episode["total_reward"])
        completed_rewards.append(episode["completed_actions"] / 6.0)  # normalize
        api_rewards.append(episode["api_success"] / 8.0)  # normalize

    return {
        "prompt_ids": episode_prompt_ids,
        "completion_ids": episode_completion_ids,
        "logprobs": episode_logprobs,
        "total_reward": total_rewards,
        "completed_reward": completed_rewards,
        "api_reward": api_rewards,
    }

## 9. Reward Functions (Multi-component)

In [ ]:
def reward_total(completions, **kwargs):
    rewards = kwargs.get("total_reward") if kwargs else None
    return [float(r) for r in rewards] if rewards else [0.0 for _ in completions]

def reward_completed(completions, **kwargs):
    rewards = kwargs.get("completed_reward") if kwargs else None
    return [float(r) for r in rewards] if rewards else [0.0 for _ in completions]

def reward_api_success(completions, **kwargs):
    rewards = kwargs.get("api_reward") if kwargs else None
    return [float(r) for r in rewards] if rewards else [0.0 for _ in completions]

## 10. Create Dataset and GRPO Config

In [ ]:
from datasets import Dataset
from trl import GRPOConfig

dataset_size = 500
dataset_prompt = "Orchestrate the enterprise workflow."
dataset = Dataset.from_dict({"prompt": [dataset_prompt] * dataset_size})

output_dir = "workflow-arena-Qwen3-1.7B"

grpo_config = GRPOConfig(
    num_train_epochs=1,
    learning_rate=5e-6,
    gradient_accumulation_steps=32,
    per_device_train_batch_size=1,
    warmup_steps=10,
    num_generations=2,
    max_completion_length=512,
    max_prompt_length=2000,
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.1,
    output_dir=output_dir,
    report_to="trackio",
    trackio_space_id=output_dir,
    logging_steps=1,
    save_steps=10,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    push_to_hub=True,
)

## 11. Initialize Trainer and Train

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model_name,
    processing_class=tokenizer,
    reward_funcs=[reward_total, reward_completed, reward_api_success],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)

In [ ]:
# Track rewards for plotting
import matplotlib.pyplot as plt

training_rewards = []

# Patch the rollout_func to track rewards
original_rollout = rollout_func
def tracked_rollout(prompts, trainer=None):
    result = original_rollout(prompts, trainer)
    training_rewards.extend(result["total_reward"])
    return result
trainer.rollout_func = tracked_rollout

# Train
trainer_stats = trainer.train()

print(f"Training runtime: {trainer_stats.metrics['train_runtime']:.1f}s")

## 12. Plot Reward Improvement (KEY DEMO CHART)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Compute rolling average for smoother curve
window = 5
if len(training_rewards) >= window:
    rolling = [np.mean(training_rewards[max(0, i - window + 1):i + 1]) for i in range(len(training_rewards))]
else:
    rolling = training_rewards

plt.figure(figsize=(12, 5))
plt.plot(training_rewards, alpha=0.3, label="Raw reward")
plt.plot(rolling, linewidth=2, label=f"Rolling avg (window={window})")
plt.xlabel("Episode", fontsize=12)
plt.ylabel("Total Reward", fontsize=12)
plt.title("WorkFlow Arena: Agent Learning to Orchestrate Enterprise Workflows", fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Baseline (episode 1): {training_rewards[0]:.3f}")
print(f"After training (last 10 avg): {np.mean(training_rewards[-10:]):.3f}")
print(f"Peak reward: {max(training_rewards):.3f}")

## 13. Save and Push to Hub

In [ ]:
client.close()
trainer.save_model(output_dir)
trainer.push_to_hub()

## 14. Before/After Comparison (for the demo)

In [ ]:
from transformers import AutoModelForCausalLM

print("Loading trained model...")
trained_model = AutoModelForCausalLM.from_pretrained(
    output_dir, dtype="auto", device_map="auto"
)

def try_workflow(model, task_name):
    client = WorkFlowClient()
    result = client.reset(task_name)
    observation = result["observation"]["content"]
    total_reward = 0
    completed = 0

    for _ in range(6):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": observation[:3000]},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False, enable_thinking=False,
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=400, temperature=0.3)
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        step = client.step(response)
        total_reward += float(step.get("reward", 0) or 0)
        completed = step["info"]["completed_actions"]
        if step["done"]:
            break
        observation = step["observation"]["content"]
    client.close()
    return total_reward, completed

for task in ["employee_onboarding", "expense_approval"]:
    reward, completed = try_workflow(trained_model, task)
    print(f"{task}: reward={reward:.3f}, completed={completed} actions")

## 🎉 Done!

You now have:
1. A trained WorkFlow Arena agent
2. A reward improvement curve (saved as `reward_curve.png`)
3. Model pushed to HuggingFace Hub
4. Before/after comparison metrics

Use these in your 3-minute pitch to show the **20% reward improvement criterion**.